**BGE-M3 Embedding Üretimi, Qdrant İndeksleme ve Hybrid Retrieval (BM25 + Dense + RRF) Testi**

Bu adımda:

size_1000_overlap_0 deneyindeki 2.193 mevzuat pasajı için GPU üzerinde BAAI/bge-m3 (1024 boyutlu) dense vektörler üretilir.
Vektörler yerel Qdrant veritabanına (isg_size_1000_overlap_0) indekslenir.
Örnek İSG soruları üzerinde BM25, Dense ve Hybrid (Reciprocal Rank Fusion) arama sonuçları karşılaştırılır.

In [ ]:
%cd /content/SLAI
!pip install -U nvidia-nccl-cu12
!python -m embeddings.generate_embeddings --experiment size_1000_overlap_0
!python -m vectorstore.index_all_experiments
!python -m retrieval.run_hybrid_demo

/content/SLAI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 336.3/336.5 MB 140.4 MB/s eta 0:00:01
ERROR: Operation cancelled by user


KeyboardInterrupt: 

## Deneysel Bulgular Özeti (BM25 vs Dense vs Hybrid RRF)
- **Soru 1 (İSG Kurulu çalışan sayısı):** Her üç yöntem de `6331 | MADDE 22`'yi 1. sırada bulmuştur. BM25 3. sırada mülga `4857 | Madde 80`'i getirirken, Hybrid arama mülga maddeyi eleyip yürürlükteki yönetmelik maddelerini (`7.5.16924 | MADDE 10-11`) ilk 3'e taşımıştır.
- **Soru 2 (İSG eğitimlerinin yenilenme sıklığı):** BM25 yalnızca yönetmeliği (`7.5.45087 | MADDE 5`), Dense ise öncelikle kanunu (`6331 | MADDE 17`) öne çıkarırken; **Hybrid (RRF)** her iki kaynağı da ilk 2 sıraya yerleştirmiştir.
- **Soru 3 (Çok tehlikeli sınıfta iş güvenliği uzmanı çalışma süresi):** BM25 kelime benzerliği nedeniyle *Diğer sağlık personeli* maddesini (`7.5.18615 | MADDE 19`) 1. sıraya koyarak yanılırken; **Dense ve Hybrid (RRF)** doğru madde olan *İş güvenliği uzmanlarının çalışma sürelerini* (`7.5.16923 | MADDE 12`) 1. sıraya taşımıştır.


oluşan verileri zipe kaydeder

In [ ]:
!zip -r -q isg_veriler_1000_0.zip data/processed data/chunks/size_1000_overlap_0 data/embeddings/size_1000_overlap_0 data/qdrant
from google.colab import files
files.download("isg_veriler_1000_0.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# LLM Cevap Üretimi (Zeynep)

**Ne yapar:** Retrieval'ın bulduğu ilk 5 pasajı (BM25 / Dense / Hybrid) Qwen3-8B'ye verir, pasajlara dayanan ve kaynak gösteren cevap üretir, sonuçları `.jsonl` dosyasına kaydeder.

**Girdi:** `results/retrieval/<deney>/llm_contexts.json` (repoda, Evrim üretiyor)
**Çıktı:** `/content/answers_<deney>_<prompt_surumu>.jsonl` (her satır = 1 soru × 1 yöntem)

---
## Çalıştırma rehberi

| # | Bölüm | Ne zaman çalıştırılır |
|---|-------|----------------------|
| 1 | Kurulum | **Her yeni oturumda** (Colab sıfırlanınca kütüphaneler silinir) |
| 2 | Model yükleme | **Her yeni oturumda** (GPU gerekir, birkaç dakika sürer) |
| 3 | Ayarlar | **Her oturumda**, ayar değişince tekrar |
| 4 | Sistem promptu | **Her oturumda**, prompt değişince tekrar |
| 5 | Fonksiyonlar | **Her oturumda** (çıktı vermez, sorun değil) |
| 6 | Repo / veri | **Her yeni oturumda** (`/content/SLAI` yoksa indirir, varsa günceller) |
| 7 | Hızlı test | İsteğe bağlı, sadece modelin çalıştığını görmek için |
| 8 | **Toplu üretim** | Soru seti veya prompt değişince. Yarıda kesilirse tekrar çalıştır, kaldığı yerden devam eder |
| 9 | Özet + indirme | Her üretimden sonra, dosyayı **mutlaka indir** (Colab diski kapanınca silinir) |
| 10 | Canlı retrieval | **Opsiyonel**, sadece arayüz / canlı soru için. Henüz denenmedi |

**Yeni oturum = 1 → 6 sırayla, sonra 8.**

**Colab ne zaman sıfırlanır?** Bağlantı kopunca, uzun süre boşta kalınca, GPU değişince. "Oturumu yeniden başlat" dosyaları silmez, sadece Python'u sıfırlar (kütüphaneler ve `/content` dosyaları kalır, ama değişkenler ve yüklü model gider).

**Ekip için kural:** GPU'yu sadece asıl çalıştırmada aç, işin bitince *Çalışma zamanı → Çalışma zamanı bağlantısını kes ve sil* yap.

## 1. Kurulum  `[HER YENİ OTURUMDA]`

Bu hücreyi çalıştırdıktan sonra uyarı çıkarsa Çalışma zamanı → Oturumu yeniden başlat yap, sonra bu hücreyi atlayıp 2'den devam et.

In [ ]:
import os
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

if not os.path.exists("/content/SLAI"):
    !git clone https://{token}@github.com/yusufdinc974/SLAI.git /content/SLAI
else:
    !cd /content/SLAI && git pull

%cd /content/SLAI
!pip install -q -r requirements.txt python-docx
!pip install -q -U "transformers>=4.51" accelerate bitsandbytes

import importlib.metadata as md
print("transformers:", md.version("transformers"))
print("bitsandbytes:", md.version("bitsandbytes"))   # >= 0.46.1 olmalı

Cloning into '/content/SLAI'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 183 (delta 67), reused 163 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (183/183), 1.57 MiB | 18.94 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/SLAI
transformers: 5.22.2
bitsandbytes: 0.50.2


## 2. QWen3-8B Model yükleme  `[HER YENİ OTURUMDA]`

GPU açık olmalı (*Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU*). Qwen3-8B 4-bit yüklenir, yaklaşık 6 GB VRAM tutar.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen3-8B"

# Tokenizer yükleme
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# 4-bit Kuantizasyon Yapılandırması
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Modeli GPU'ya yükleme
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model ve Tokenizer başarıyla yüklendi!")
tok = tokenizer

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model ve Tokenizer başarıyla yüklendi!


model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model ve Tokenizer başarıyla yüklendi!


## 3. Ayarlar  `[HER OTURUMDA]`
BM25, Dense ve Hybrid karşılaştırmasında **model, prompt, `THINKING`, `TOP_K` ve `MAX_NEW_TOKENS` aynı kalmalı**, yoksa karşılaştırma geçersiz olur.

Prompt'u değiştirirsen `PROMPT_VERSION`'ı da değiştir (v3, v4...), böylece eski çıktı dosyası üzerine yazılmaz.

In [ ]:
EXPERIMENT     = "size_1000_overlap_100"   # hangi chunk deneyi
PROMPT_VERSION = "v2"                      # prompt değişince v3 yap
THINKING       = False                     # Qwen3 düşünme modu (final deneyde sabit)
METHODS        = ["bm25", "dense", "hybrid"]
TOP_K          = 5
MAX_NEW_TOKENS = 512

REPO_DIR     = "/content/SLAI"
CONTEXT_PATH = f"{REPO_DIR}/results/retrieval/{EXPERIMENT}/llm_contexts.json"
OUT          = f"/content/answers_{EXPERIMENT}_{PROMPT_VERSION}{'_think' if THINKING else ''}.jsonl"

RED = "Sağlanan mevzuat metninde bu sorunun cevabı bulunmamaktadır."   # "cevap yok" sabit cümlesi
print("Çıktı dosyası:", OUT)

Çıktı dosyası: /content/answers_size_1000_overlap_100_v2.jsonl


## 4. Sistem promptu  `[HER OTURUMDA, prompt değişince tekrar]`

In [ ]:
SYSTEM_PROMPT = """Sen bir İş Sağlığı ve Güvenliği (İSG) mevzuat asistanısın.
Sana mevzuat pasajları ve bir soru ya da olay anlatımı verilecek. Türkçe yanıtla.

Kurallar:
1. Hukuki dayanak olarak yalnızca verilen pasajları kullan. Pasajlarda olmayan bir kural, süre, ceza veya yükümlülük ekleme.
2. Soru bir olay ya da senaryo ise, pasajlardaki ilgili kuralları olaya uygulayarak cevapla. Önce hangi maddenin neden geçerli olduğunu, sonra sonucu yaz.
3. Pasajlar sorunun cevabı için gerekli kuralı hiç içermiyorsa cevabın tamamı yalnızca şu cümle olsun, başka hiçbir şey yazma (Kaynak satırı dahil): "Sağlanan mevzuat metninde bu sorunun cevabı bulunmamaktadır."
4. Soru belirli bir süre, sayı, oran veya miktar soruyorsa ve pasajlarda bu değer yoksa bunu açıkça belirt (örneğin "Pasajlarda belirli bir süre öngörülmemiştir.") ve pasajlarda yazanı ekle.
5. Pasajlar cevabın yalnızca bir kısmını içeriyorsa, bulunan kısmı yaz ve eksik kısmın pasajlarda olmadığını belirt.
6. Sadece cevabı destekleyen pasajlara atıf yap, ilgisiz pasajları kullanma.
7. Cevabın sonuna yeni bir satırda, yalnızca cevabı destekleyen pasajların Pasaj ID değerlerini şu biçimde yaz (köşeli parantez, madde numarası veya başlık kullanma):
Kaynak: id1, id2
8. Markdown (kalın yazı, başlık) kullanma.
9. Kanunun veya yönetmeliğin adını yalnızca pasajlarda geçiyorsa yaz, kendi bilginle ekleme. Bir pasajdaki bilgiyi, o pasajda adı geçmeyen bir kanuna veya yönetmeliğe atfetme.
10. Cümlenin öznesini (işveren, çalışan, kurul vb.) pasajdaki gibi koru, yükümlülüğü başka bir özneye kaydırma.
"""

## 5. Fonksiyonlar  `[HER OTURUMDA]`  (çıktı vermez)

In [ ]:
import json, os, re

def prep(passages):
    return [{"passage_id": p["passage_id"], "text": p.get("text", "")} for p in passages]

def generate_rag_response(query, passages, thinking=None, return_thinking=False):
    thinking = THINKING if thinking is None else thinking
    context_text = ""
    for p in passages:
        context_text += f"\n[Pasaj ID: {p['passage_id']}]\n{p['text']}\n"

    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Bağlam:\n{context_text}\n\nSoru: {query}"},
    ]
    inputs = tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", enable_thinking=thinking
    )
    input_ids = (inputs.to(model.device) if isinstance(inputs, torch.Tensor)
                 else inputs["input_ids"].to(model.device))

    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=2048 if thinking else MAX_NEW_TOKENS,
        do_sample=False,
    )
    text = tok.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)

    if "</think>" in text:
        think, answer = text.split("</think>", 1)
        think, answer = think.replace("<think>", "").strip(), answer.strip()
    else:
        think, answer = "", text.strip()
    return (answer, think) if return_thinking else answer

def clean_answer(text):
    text = text.replace("**", "").strip()
    return RED if RED in text else text

def extract_citations(answer, passage_ids):
    m = re.search(r"Kaynak:(.*)", answer, flags=re.S)
    satir = m.group(1) if m else ""
    return [pid for pid in dict.fromkeys(passage_ids) if pid in satir]

print("Fonksiyonlar hazır.")

### Mock Kanun Verisiyle Test

In [ ]:
    # MOCK (Sahte) Retrieval Verisi — Kişi 1 & 2'den bu formatta veri gelecek
mock_passages = [
    {
        "passage_id": "6331_Madde_13_Fikra_1",
        "text": "Ciddi ve yakın tehlike ile karşı karşıya kalan çalışanlar kurula, kurulun bulunmadığı işyerlerinde ise işverene başvurarak durumun tespit edilmesini ve gerekli tedbirlerin alınmasına karar verilmesini talep edebilir."
    }
]

# MOCK (Sahte) Retrieval Verisi — Kişi 1 & 2'den bu formatta veri gelecek
mock_passages = [
    {
        "passage_id": "6331_Madde_13_Fikra_1",
        "text": "Ciddi ve yakın tehlike ile karşı karşıya kalan çalışanlar kurula, kurulun bulunmadığı işyerlerinde ise işverene başvurarak durumun tespit edilmesini ve gerekli tedbirlerin alınmasına karar verilmesini talep edebilir."
    }
]

# Test 1: Bağlamda cevabı OLAN soru
query_1 = "İSG kurulu bulunmayan bir işyerinde tehlike gören çalışan ilk olarak nereye başvurmalıdır?"
print("--- TEST 1 (Cevap Bağlamda Var) ---")
print(generate_rag_response(query_1, mock_passages))

# Test 2: Bağlamda cevabı OLMAYAN soru (Hallucination Testi)
# THINKING
query_2 = "İşverenin yıllık vermekle yükümlü olduğu kıdem tazminatı tavanı ne kadardır?"
print("\n--- TEST 2 (Cevap Bağlamda Yok) ---")
ans, think = generate_rag_response(query_2, mock_passages, thinking=True, return_thinking=True)
print("CEVAP:\n", ans)
print("\nDÜŞÜNME:\n", think)

--- TEST 1 (Cevap Bağlamda Var) ---
İSG kurulu bulunmayan bir işyerinde tehlike gören çalışan, ilk olarak işverene başvurmalıdır.

Kaynak: 6331_Madde_13_Fikra_1

--- TEST 2 (Cevap Bağlamda Yok) ---
CEVAP:
 Sağlanan mevzuat metninde bu sorunun cevabı bulunmamaktadır.  
Kaynak: 6331_Madde_13_Fikra_1

DÜŞÜNME:
 Okay, the user is asking about the maximum amount of annual seniority allowance that the employer is obligated to pay. Let me check the provided passage.

The given passage is from Pasaj ID 6331_Madde_13_Fikra_1, which talks about employees facing serious and imminent danger being able to request the employer or the committee to take necessary measures. There's no mention of seniority allowance or its maximum limit here. 

Since the question is about the cap on annual seniority allowance, and the provided passage doesn't cover that, I need to state that the information isn't available in the given text. I should also note that the answer isn't present in the provided materials.


# LLM Pipeline

Drivedan data ve embedding ziplerini çekip colaba çıkarma

In [ ]:
# gdown kütüphanesi ile Google Drive'daki büyük zipleri indirme
!pip install -q gdown

# Drive dosya ID'lerinizi buraya yazın (paylaşım linkindeki 'id=' kısmı)
DATA_ZIP_ID = "1osBUbfOkTkhB4SEdm_qECo2OskoV-yNE"
EMBEDDINGS_ZIP_ID = "1-r_IFhaNjpNF6kT1hQ7tm7xwDlgVwJLv"

# Dosyaları indiriyoruz
!gdown --id {DATA_ZIP_ID} -O /content/data.zip
!gdown --id {EMBEDDINGS_ZIP_ID} -O /content/embeddings.zip

# Zipleri açıyoruz
!unzip -q -o /content/data.zip -d /content/
!unzip -q -o /content/embeddings.zip -d /content/

print("Veriler ve indeksler başarıyla indirildi ve açıldı!")

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1osBUbfOkTkhB4SEdm_qECo2OskoV-yNE
From (redirected): https://drive.google.com/uc?id=1osBUbfOkTkhB4SEdm_qECo2OskoV-yNE&confirm=t&uuid=5909af8a-4d49-4420-9d9f-ded1de7305e7
To: /content/data.zip
100% 199M/199M [00:01<00:00, 133MB/s]
/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1-r_IFhaNjpNF6kT1hQ7tm7xwDlgVwJLv
From (redirected): https://drive.google.com/uc?id=1-r_IFhaNjpNF6kT1hQ7tm7xwDlgVwJLv&confirm=t&uuid=c75608f8-6dcc-444c-af8a-30ea8d124105
To: /content/embeddings.zip
100%

## Retrieval

In [ ]:
import json

path = "/content/SLAI/results/retrieval/size_1000_overlap_100/llm_contexts.json"
data = json.load(open(path, encoding="utf-8"))
print(len(data), "soru var")
print(data[0]["question"])

def prep(passages):
    return [{"passage_id": p["passage_id"], "text": p.get("text", "")} for p in passages]

item = data[0]
for method in ["bm25", "dense", "hybrid"]:
    print("\n====", method.upper(), "====")
    print(generate_rag_response(item["question"], prep(item["methods"][method])))

4 soru var
İş sağlığı ve güvenliği kurulu kaç çalışan olan işyerlerinde kurulmalıdır?

==== BM25 ====
İş sağlığı ve güvenliği kurulu, elli ve daha fazla çalışanın bulunduğu ve altı aydan fazla süren sürekli işlerin yapıldığı işyerlerinde kurulmalıdır.

Kaynak: [Pasaj ID: 1988af9deabab96ce994]

==== DENSE ====
Elli ve daha fazla çalışanın bulunduğu ve altı aydan fazla süren sürekli işlerin yapıldığı işyerlerinde işveren, iş sağlığı ve güvenliği ile ilgili çalışmalarda bulunmak üzere kurul oluşturur.

Kaynak: 1988af9deabab96ce994, e44336281951ef871f64

==== HYBRID ====
İş sağlığı ve güvenliği kurulu, elli ve daha fazla çalışanın bulunduğu ve altı aydan fazla süren sürekli işlerin yapıldığı işyerlerinde kurulmalıdır.

Kaynak: 1988af9deabab96ce994


test

In [ ]:
import re, json

RED = "Sağlanan mevzuat metninde bu sorunun cevabı bulunmamaktadır."

def extract_citations(answer, passage_ids):
    m = re.search(r"Kaynak:(.*)", answer, flags=re.S)
    satir = m.group(1) if m else ""
    return [pid for pid in dict.fromkeys(passage_ids) if pid in satir]

for item in data:
    for method in ["bm25", "dense", "hybrid"]:
        passages = item["methods"][method]
        pids = [p["passage_id"] for p in passages]
        raw = generate_rag_response(item["question"], prep(passages))
        cited = extract_citations(raw, pids)
        rec = {
            "experiment": "size_1000_overlap_100",
            "question_id": item["question_id"],
            "method": method,
            "question": item["question"],
            "passage_ids": pids,
            "raw_answer": raw,
            "answer": RED if RED in raw else raw,
            "refused": RED in raw,
            "cited_ids": cited,
            "citation_ok": bool(cited) or RED in raw,
        }
        with open(OUT, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        print(item["question_id"], method, "tamam")

q001 bm25 tamam
q001 dense tamam
q001 hybrid tamam
q002 bm25 tamam
q002 dense tamam
q002 hybrid tamam
q003 bm25 tamam
q003 dense tamam
q003 hybrid tamam
q004 bm25 tamam
q004 dense tamam
q004 hybrid tamam


In [ ]:
from google.colab import files
files.download(OUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
for line in open(OUT, encoding="utf-8"):
    r = json.loads(line)
    print("=" * 70)
    print(r["question_id"], "|", r["method"].upper(), "| ret:", r["refused"])
    print("SORU:", r["question"])
    print("CEVAP:", r["answer"])

q001 | BM25 | ret: False
SORU: İş sağlığı ve güvenliği kurulu kaç çalışan olan işyerlerinde kurulmalıdır?
CEVAP: İş sağlığı ve güvenliği kurulu, elli ve daha fazla çalışanın bulunduğu ve altı aydan fazla süren sürekli işlerin yapıldığı işyerlerinde kurulmalıdır.

Kaynak: 1988af9deabab96ce994
q001 | DENSE | ret: False
SORU: İş sağlığı ve güvenliği kurulu kaç çalışan olan işyerlerinde kurulmalıdır?
CEVAP: 50 ve daha fazla çalışanın bulunduğu işyerlerinde iş sağlığı ve güvenliği kurulu kurulmalıdır.

Kaynak: 1988af9deabab96ce994, e44336281951ef871f64
q001 | HYBRID | ret: False
SORU: İş sağlığı ve güvenliği kurulu kaç çalışan olan işyerlerinde kurulmalıdır?
CEVAP: İş sağlığı ve güvenliği kurulu, elli ve daha fazla çalışanın bulunduğu ve altı aydan fazla süren sürekli işlerin yapıldığı işyerlerinde kurulmalıdır.

Kaynak: 1988af9deabab96ce994
q002 | BM25 | ret: False
SORU: İşveren çalışanların iş sağlığı ve güvenliği eğitimlerini hangi sıklıkla yenilemek zorundadır?
CEVAP: İşveren, çalışanla

In [ ]:
for qid in ["q002", "q004"]:
    item = next(d for d in data if d["question_id"] == qid)
    print("#####", qid, item["question"])
    for p in item["methods"]["hybrid"]:
        print(p["passage_id"], "\n", p["text"][:600], "\n")

##### q002 İşveren çalışanların iş sağlığı ve güvenliği eğitimlerini hangi sıklıkla yenilemek zorundadır?
5da4b7cf3c7ea8efca2c 
 Çalışanların eğitimi
MADDE 17 – (1) İşveren, çalışanların iş sağlığı ve güvenliği eğitimlerini almasını sağlar. Bu eğitim özellikle; işe başlamadan önce, çalışma yeri veya iş değişikliğinde, iş ekipmanının değişmesi hâlinde veya yeni teknoloji uygulanması hâlinde verilir. Eğitimler, değişen ve ortaya çıkan yeni risklere uygun olarak yenilenir, gerektiğinde ve düzenli aralıklarla tekrarlanır.
(2) Çalışan temsilcileri özel olarak eğitilir.
(3) Mesleki eğitim alma zorunluluğu bulunan tehlikeli ve çok tehlikeli sınıfta yer alan işlerde, yapacağı işle ilgili mesleki eğitim aldığını belgeleyemeyenle 

aef02d0c95dfeb81b6e2 
 İşverenin yükümlülükleri
MADDE 5- (1) İşveren, çalışanların işe başlama eğitimlerini, temel eğitimleri ve gerekli hâllerde bu Yönetmelikte yer alan diğer eğitimleri almasını sağlar.
(2) İşveren, çalışanların iş sağlığı ve güvenliği eğitimlerine 

## 8. Toplu üretim  `[SORU SETİ / PROMPT DEĞİŞİNCE]`
Her soru için her yöntemin (BM25, Dense, Hybrid) ilk `TOP_K` pasajıyla cevap üretir ve **her cevabı hemen dosyaya yazar**

Colab kopsa bile tekrar çalıştırınca daha önce üretilenleri atlar. Baştan başlamak istersen `OUT` dosyasını sil: `os.remove(OUT)`

In [ ]:
done = set()
if os.path.exists(OUT):
    for line in open(OUT, encoding="utf-8"):
        r = json.loads(line)
        done.add((r["question_id"], r["method"]))
    print(len(done), "cevap zaten var, atlanacak")

toplam = len(data) * len(METHODS)
for item in data:
    for method in METHODS:
        if (item["question_id"], method) in done:
            continue
        passages = item["methods"][method][:TOP_K]
        pids = [p["passage_id"] for p in passages]

        raw = generate_rag_response(item["question"], prep(passages))
        cited = extract_citations(raw, pids)
        rec = {
            "experiment": EXPERIMENT,
            "prompt_version": PROMPT_VERSION,
            "thinking": THINKING,
            "model": MODEL,
            "question_id": item["question_id"],
            "method": method,
            "question": item["question"],
            "passage_ids": pids,
            "raw_answer": raw,                       # modelin ham çıktısı (analiz için sakla)
            "answer": clean_answer(raw),             # temizlenmiş cevap
            "refused": RED in raw,
            "cited_ids": cited,
            "citation_ok": bool(cited) or RED in raw,
        }
        with open(OUT, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        done.add((item["question_id"], method))
        print(f"[{len(done)}/{toplam}]", item["question_id"], method)
print("Bitti.")

## 9. Özet ve indirme  `[HER ÜRETİMDEN SONRA]`
Dosyayı indirmeyi unutma, Colab diski oturum kapanınca silinir.

In [ ]:
rows = [json.loads(l) for l in open(OUT, encoding="utf-8")]

print("YÖNTEM   | cevap | ret | atıf yok")
for m in METHODS:
    rm = [r for r in rows if r["method"] == m]
    print(f"{m:8} | {len(rm):5} | {sum(r['refused'] for r in rm):3} | {sum(not r['citation_ok'] for r in rm)}")

for r in rows:
    print("=" * 70)
    print(r["question_id"], "|", r["method"].upper(), "| ret:", r["refused"])
    print("SORU:", r["question"])
    print("CEVAP:", r["answer"])